# Prepare one Uniswap v2 dataset

This optional companion to `capstone_amm_study.ipynb` uses only Python's standard library. It extracts a single pool's events from an archive-capable Ethereum JSON-RPC endpoint, pairs Sync with Swap in log order, reconstructs ordinary swaps, and downloads matching completed one-minute reference candles. Alternatively, use a previously saved raw export with its checkpoint and metadata. No RPC credential is printed or written into the dataset card.

The fixed pool, token order, quote currency, eight-week window and reference symbol are configured below. The compact submission includes the prepared actual dataset; larger raw responses are supplied separately. A new acquisition should use a fresh directory. The last section runs small **synthetic fixtures**, labels them as fixtures, and never presents them as market evidence. RPC/network acquisition requires access to the chosen provider; the offline reconstruction checks require no external packages or network.

References: [Ethereum JSON-RPC](https://ethereum.org/developers/docs/apis/json-rpc/), [Uniswap v2 contract](https://github.com/Uniswap/v2-core/blob/master/contracts/UniswapV2Pair.sol), and [Binance public archive format](https://github.com/binance/binance-public-data).


In [1]:
import csv, hashlib, io, json, math, os, random, sqlite3, tempfile, time, urllib.request, urllib.error, zipfile
from bisect import bisect_left
from collections import Counter
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.parse import urlsplit

# Network acquisition is explicit; ordinary notebook/fixture runs stay offline.
ACQUIRE = os.environ.get("AMM_ACQUIRE", "0") == "1"
if ACQUIRE and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.removeprefix("export ").split("=", 1)
            if key.strip() in ("ALCHEMY_API_KEY", "ALCHEMY_URL", "AMM_RPC_URL"):
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
RPC_URL = os.environ.get("AMM_RPC_URL") or os.environ.get("ALCHEMY_URL", "")
POOL_ADDRESS = "0xb4e16d0168e52d35cacd2c6185b44281ec28c9dc"
START_UTC, END_UTC = "2024-01-01T00:00:00+00:00", "2024-02-26T00:00:00+00:00"  # End exclusive.
START_BLOCK, END_BLOCK = None, None  # Resolved once by block timestamp, then saved.
BASE_TOKEN_INDEX = 1
QUOTE_SYMBOL, REFERENCE_SYMBOL = "USDC", "ETHUSDC"
REFERENCE_DELAY_SECONDS = 1.0
DATA_DIR = Path(os.environ.get("AMM_DATA_DIR", "real_data/uniswap_v2_usdc_weth_2024_8weeks"))
FIELDS = ["timestamp","origin_timestamp","block_number","transaction_index","log_index",
          "transaction_hash","segment","reserve0_before","reserve1_before","reserve0_after",
          "reserve1_after","amount0In","amount1In","amount0Out","amount1Out"]
RPC_CU_PER_SECOND = 280  # Below the documented 300 CU/s free-tier limit.
RPC_COST = {"eth_getLogs": 60, "eth_call": 26, "eth_chainId": 5}
RPC_NEXT = 0.; RPC_USAGE = Counter()

def rpc_many(calls):
    global RPC_NEXT
    allowed = {"eth_getLogs", "eth_call", "eth_chainId", "eth_getBlockByNumber", "web3_sha3"}
    if any(method not in allowed for method, _ in calls): raise ValueError("Only study read methods are allowed")
    cost = sum(RPC_COST.get(method, 20) for method, _ in calls)
    payload = [dict(jsonrpc="2.0", id=i, method=m, params=p) for i,(m,p) in enumerate(calls)]
    for attempt in range(7):
        time.sleep(max(0., RPC_NEXT-time.monotonic()))
        RPC_NEXT = time.monotonic()+cost/RPC_CU_PER_SECOND
        req = urllib.request.Request(RPC_URL, data=json.dumps(payload).encode(), headers={"Content-Type":"application/json"})
        try:
            with urllib.request.urlopen(req, timeout=90) as response: result = json.load(response)
        except (urllib.error.URLError, TimeoutError, OSError, json.JSONDecodeError) as exc:
            status = getattr(exc, "code", None)
            if status in (400,401,403): raise RuntimeError(f"RPC access/configuration failed (HTTP {status}); credentials withheld") from None
            if attempt == 6: raise RuntimeError(f"RPC transport failed ({type(exc).__name__}); credentials withheld") from None
            time.sleep(min(30, 2**attempt)); RPC_USAGE["retries"] += 1; continue
        if not isinstance(result, list): raise RuntimeError("Expected a JSON-RPC batch response")
        by_id = {item["id"]:item for item in result}
        if set(by_id) != set(range(len(calls))): raise RuntimeError("Incomplete RPC batch")
        errors = [item["error"] for item in result if "error" in item]
        if errors:
            codes = {e.get("code") for e in errors}
            if codes <= {429,-32005} and attempt < 6:
                time.sleep(min(30,2**attempt)); RPC_USAGE["retries"] += 1; continue
            raise RuntimeError(f"RPC methods failed with codes {sorted(codes)}; provider messages/credentials withheld")
        RPC_USAGE.update(method for method,_ in calls)
        RPC_USAGE["estimated_billing_cu"] += sum(0 if m=="eth_chainId" else RPC_COST.get(m,20) for m,_ in calls)
        return [by_id[i]["result"] for i in range(len(calls))]

def rpc(method, params):
    return rpc_many([(method,params)])[0]

def words(data):
    value=data.removeprefix("0x")
    if len(value)%64: raise ValueError("Malformed ABI data")
    return [int(value[i:i+64],16) for i in range(0,len(value),64)]

def write_csv(path, rows, fields):
    with Path(path).open("w", newline="") as f:
        writer=csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows(rows)

def first_block_at(utc):
    target=int(datetime.fromisoformat(utc).timestamp())
    low,high=18_000_000,20_000_000  # Bracket covers the fixed 2024 window.
    if not (int(rpc("eth_getBlockByNumber",[hex(low),False])["timestamp"],16)<target<=
            int(rpc("eth_getBlockByNumber",[hex(high),False])["timestamp"],16)):
        raise ValueError("Timestamp outside configured search bracket")
    while low<high:
        middle=(low+high)//2
        if int(rpc("eth_getBlockByNumber",[hex(middle),False])["timestamp"],16)<target: low=middle+1
        else: high=middle
    return low


## Acquire and save immutable source records

The fixed window is January 1–February 25, 2024 (eight complete UTC weeks), for the Ethereum mainnet Uniswap v2 USDC/WETH pair. Run with `AMM_ACQUIRE=1` to read `ALCHEMY_URL` from the local `.env` (or set `AMM_RPC_URL` in Colab secrets/environment). Never put a key in a notebook cell. The free tier permits ten blocks per log query. Read-only requests are batched, paced below 300 CU/s, and retried on transient failures. A SQLite cache records **every** queried range, including empty responses, so an interrupted run resumes without downloading completed ranges again.

All pool logs are requested, so a reserve change without an ordinary swap creates a segment boundary. Initial reserves come from `getReserves()` immediately before the window. Token order, decimals, and chain ID are checked. Alchemy supplies `blockTimestamp` and `blockHash` in each raw log; these are checked for within-block consistency and against a fetched header at least once per UTC day. Providers without timestamps fall back to fetching every needed header. The raw records, range ledger, checkpoint, metadata and hashes are retained. Only the provider hostname is saved; the endpoint/key is never saved.

Re-running a completed, checksum-verified export reuses it. To change the fixed window, use a new directory. A three-day pilot checks reconciliation before the same extraction continues over the full frozen window; it is not used for model selection.

For optional fresh acquisition in Colab, first read a saved secret into the environment (the secret value must never appear in source or output):

```python
import os
from google.colab import userdata
os.environ["AMM_RPC_URL"] = userdata.get("ALCHEMY_URL")
os.environ["AMM_ACQUIRE"] = "1"
os.environ["AMM_DATA_DIR"] = "real_data/fresh_uniswap_export"
```

The main study can run entirely from its prepared bundled inputs without a key or a new download. This notebook's saved outputs include the completed actual export and, separately, explicitly labeled offline fixtures.


In [2]:
def acquire_export(destination, chunk=10, stop_block=None):
    global START_BLOCK, END_BLOCK
    if not RPC_URL or not POOL_ADDRESS: raise ValueError("Set the provider and pool first")
    if not 1<=chunk<=10: raise ValueError("Use at most ten blocks on this free-tier protocol")
    destination=Path(destination); destination.mkdir(parents=True,exist_ok=True)
    card=destination/"dataset_card.json"
    if card.exists():
        meta=json.loads(card.read_text()); START_BLOCK,END_BLOCK=meta["start_block"],meta["end_block"]
        if (meta["pool_address"]!=POOL_ADDRESS.lower() or meta["start_utc"]!=START_UTC or meta["end_utc_exclusive"]!=END_UTC
            or meta["query"]["chunk"]!=chunk): raise ValueError("Existing export uses another frozen source configuration")
        if (destination/"checksums.json").exists():
            hashes=json.loads((destination/"checksums.json").read_text())
            for name,digest in hashes.items():
                if hashlib.sha256((destination/name).read_bytes()).hexdigest()!=digest: raise ValueError(f"Frozen export changed: {name}")
            print("Reusing checksum-verified real export",flush=True); return destination
    else:
        if START_BLOCK is None: START_BLOCK=first_block_at(START_UTC)
        if END_BLOCK is None: END_BLOCK=first_block_at(END_UTC)-1
        call=lambda address,selector:rpc("eth_call",[{"to":address,"data":selector},hex(START_BLOCK-1)])
        chain=int(rpc("eth_chainId",[]),16)
        token0="0x"+call(POOL_ADDRESS,"0x0dfe1681")[-40:]; token1="0x"+call(POOL_ADDRESS,"0xd21220a7")[-40:]
        decimals=[int(call(t,"0x313ce567"),16) for t in (token0,token1)]
        if (chain,token0.lower(),token1.lower(),decimals)!=(1,"0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48","0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2",[6,18]):
            raise ValueError("Unexpected chain or USDC/WETH token configuration")
        initial=rpc("eth_getBlockByNumber",[hex(START_BLOCK-1),False])
        checkpoint=dict(reserves=words(call(POOL_ADDRESS,"0x0902f1ac"))[:2],timestamp=int(initial["timestamp"],16),block=START_BLOCK-1,block_hash=initial["hash"])
        topics={name:rpc("web3_sha3",["0x"+signature.encode().hex()]).lower() for name,signature in
                [("Sync","Sync(uint112,uint112)"),("Swap","Swap(address,uint256,uint256,uint256,uint256,address)")]}
        meta=dict(data_kind="real_market",pool_address=POOL_ADDRESS.lower(),chain_id=chain,start_block=START_BLOCK,end_block=END_BLOCK,
                  start_utc=START_UTC,end_utc_exclusive=END_UTC,base_token_index=BASE_TOKEN_INDEX,quote_symbol=QUOTE_SYMBOL,
                  reference_symbol=REFERENCE_SYMBOL,reference_delay_seconds=REFERENCE_DELAY_SECONDS,fee=.003,
                  token0=token0,token1=token1,token0_decimals=decimals[0],token1_decimals=decimals[1],topics=topics,
                  source_host=urlsplit(RPC_URL).hostname,query=dict(method="eth_getLogs",address=POOL_ADDRESS,
                  from_block=START_BLOCK,to_block=END_BLOCK,chunk=chunk),created_utc=datetime.now(timezone.utc).isoformat())
        card.write_text(json.dumps(meta,indent=2)); (destination/"checkpoint.json").write_text(json.dumps(checkpoint,indent=2))
    stop=min(END_BLOCK,stop_block if stop_block is not None else END_BLOCK)
    cache=sqlite3.connect(destination/"acquisition.sqlite3")
    cache.execute("CREATE TABLE IF NOT EXISTS ranges (start INTEGER PRIMARY KEY, end INTEGER, logs TEXT)")
    cache.execute("CREATE TABLE IF NOT EXISTS headers (block INTEGER PRIMARY KEY, payload TEXT)")
    done={row[0] for row in cache.execute("SELECT start FROM ranges")}
    starts=[s for s in range(START_BLOCK,stop+1,chunk) if s not in done]
    last_report=0.; started=time.monotonic()
    for offset in range(0,len(starts),32):
        batch=starts[offset:offset+32]
        results=rpc_many([("eth_getLogs",[{"address":POOL_ADDRESS,"fromBlock":hex(s),"toBlock":hex(min(s+chunk-1,END_BLOCK))}]) for s in batch])
        for s,logs in zip(batch,results):
            end=min(s+chunk-1,END_BLOCK)
            for record in logs:
                if record.get("removed") or not s<=int(record["blockNumber"],16)<=end or record["address"].lower()!=POOL_ADDRESS.lower():
                    raise ValueError("Invalid raw log provenance")
            cache.execute("INSERT INTO ranges VALUES (?,?,?)",(s,end,json.dumps(logs,separators=(",",":"))))
        cache.commit()
        if time.monotonic()-last_report>=60 or offset+32>=len(starts):
            total=cache.execute("SELECT COUNT(*) FROM ranges").fetchone()[0]
            progress=dict(completed_ranges=total,total_ranges=math.ceil((END_BLOCK-START_BLOCK+1)/chunk),
                          last_block=batch[-1],elapsed_seconds=round(time.monotonic()-started),session_usage=dict(RPC_USAGE))
            (destination/"acquisition_progress.json").write_text(json.dumps(progress,indent=2))
            print("Acquisition:",total,"/",progress["total_ranges"],"ranges; last block",batch[-1],flush=True); last_report=time.monotonic()
    headers={}; verify={}; fallback=set(); ledger=[]
    with (destination/"raw_logs.jsonl").open("w") as output:
        for s,end,encoded in cache.execute("SELECT start,end,logs FROM ranges ORDER BY start"):
            ledger.append(dict(from_block=s,to_block=end,sha256=hashlib.sha256(encoded.encode()).hexdigest()))
            for record in json.loads(encoded):
                block=int(record["blockNumber"],16)
                if "blockTimestamp" in record:
                    header=dict(timestamp=int(record["blockTimestamp"],16),hash=record["blockHash"])
                    if block in headers and headers[block]!=header: raise ValueError("Inconsistent block metadata")
                    headers[block]=header; verify.setdefault(header["timestamp"]//86400,block)
                else: fallback.add(block)
                output.write(json.dumps(record,separators=(",",":"))+"\n")
    needed=sorted(set(verify.values())|fallback)
    for offset in range(0,len(needed),64):
        batch=needed[offset:offset+64]; cached={row[0]:json.loads(row[1]) for row in cache.execute("SELECT block,payload FROM headers")}
        missing=[b for b in batch if b not in cached]
        received=rpc_many([("eth_getBlockByNumber",[hex(b),False]) for b in missing]) if missing else []
        for b,value in zip(missing,received):
            h=dict(timestamp=int(value["timestamp"],16),hash=value["hash"]); cached[b]=h
            cache.execute("INSERT INTO headers VALUES (?,?)",(b,json.dumps(h)))
        cache.commit()
        for b in batch:
            if b in headers and headers[b]!=cached[b]: raise ValueError("Block-header verification failed")
            headers[b]=cached[b]
    cache.close()
    complete=stop==END_BLOCK
    if complete and len(ledger)!=math.ceil((END_BLOCK-START_BLOCK+1)/chunk): raise ValueError("Incomplete query coverage")
    for name,value in [("blocks.json",headers),("query_ledger.json",ledger),
                       ("acquisition_report.json",dict(complete=complete,queried_ranges=len(ledger),header_checks=len(needed),
                        timestamp_source="RPC log blockTimestamp; sampled daily headers checked",estimated_log_cu=60*len(ledger)))]:
        (destination/name).write_text(json.dumps(value,indent=2))
    return destination

def decode_export(directory):
    directory=Path(directory); meta=json.loads((directory/"dataset_card.json").read_text())
    checkpoint=json.loads((directory/"checkpoint.json").read_text()); headers=json.loads((directory/"blocks.json").read_text())
    reverse={value:key for key,value in meta["topics"].items()}; events=[]
    for line in (directory/"raw_logs.jsonl").read_text().splitlines():
        record=json.loads(line)
        if record.get("removed"): raise ValueError("Removed source event")
        if record["address"].lower()!=meta["pool_address"].lower(): raise ValueError("Mixed pools")
        kind=reverse.get(record["topics"][0].lower()) if record.get("topics") else None
        if kind:
            block=int(record["blockNumber"],16); header=headers[str(block)]
            if record["blockHash"]!=header["hash"]: raise ValueError("Block-hash mismatch")
            events.append(dict(kind=kind,values=words(record["data"]),block_number=block,
                               transaction_index=int(record["transactionIndex"],16),log_index=int(record["logIndex"],16),
                               transaction_hash=record["transactionHash"],timestamp=header["timestamp"]))
    return meta,checkpoint,events


## Reconcile and classify events

Sync is emitted before Swap by the pair contract. Pair only within the same transaction. A non-swap Sync, a nonordinary swap, or a reconciliation failure creates a segment boundary. Retained swaps must reconcile exactly in raw integers and use the maximum-output ordinary-swap quote, including its integer rounding. Every excluded operation is recorded, so poor coverage remains visible. This narrows the empirical scope to ordinary maximum-output swaps rather than claiming every contract-valid swap has the same input-only transition.


In [3]:
def reconstruct_swaps(events, checkpoint):
    keys=lambda e:(e["block_number"],e["transaction_index"],e["log_index"])
    events=sorted(events,key=keys)
    if len({keys(e) for e in events})!=len(events): raise ValueError("Duplicate source events")
    current=tuple(map(int,checkpoint["reserves"])); previous_time=checkpoint["timestamp"]
    pending=None; segment=0; rows=[]; excluded=[]; counts=Counter()
    for event in events:
        if event["timestamp"]<previous_time: raise ValueError("Nonmonotone source timestamps")
        if event["kind"]=="Sync":
            if pending is not None:
                segment+=1; counts["non_swap_sync"]+=1
            if len(event["values"])!=2: raise ValueError("Invalid Sync payload")
            after=tuple(map(int,event["values"]))
            pending=(current,after,previous_time,event)
            current,previous_time=after,event["timestamp"]
            continue
        if event["kind"]!="Swap": continue
        if pending is None or pending[3]["transaction_hash"]!=event["transaction_hash"] or pending[3]["block_number"]!=event["block_number"]:
            raise ValueError("Swap has no immediately preceding reserve update in the same transaction")
        before,after,origin,_=pending; pending=None
        if len(event["values"])!=4: raise ValueError("Invalid Swap payload")
        i0,i1,o0,o1=map(int,event["values"]); r0,r1=before; a0,a1=after
        counts["swaps_seen"]+=1
        reason=None
        if min(r0,r1,a0,a1)<=0 or min(i0,i1,o0,o1)<0: reason="invalid_state"
        elif not ((i0>0 and o1>0 and i1==o0==0) or (i1>0 and o0>0 and i0==o1==0)): reason="nonordinary_swap"
        elif (a0,a1)!=(r0+i0-o0,r1+i1-o1): reason="reserve_accounting"
        else:
            quote=(i0*997*r1)//(1000*r0+997*i0) if i0 else (i1*997*r0)//(1000*r1+997*i1)
            if quote != (o1 if i0 else o0): reason="nonstandard_output_quote"
        if reason:
            excluded.append(dict(block_number=event["block_number"],log_index=event["log_index"],reason=reason))
            counts[reason]+=1; segment+=1; continue
        row={key:event[key] for key in ["timestamp","block_number","transaction_index","log_index","transaction_hash"]}
        row.update(origin_timestamp=origin,segment=segment,reserve0_before=r0,reserve1_before=r1,
                   reserve0_after=a0,reserve1_after=a1,amount0In=i0,amount1In=i1,amount0Out=o0,amount1Out=o1)
        rows.append(row); counts["retained"]+=1
    if pending is not None: counts["non_swap_sync"]+=1
    counts["retained_fraction"]=len(rows)/max(counts["swaps_seen"],1)
    return rows,excluded,dict(counts)

def prepare_export(directory):
    directory=Path(directory); meta,checkpoint,events=decode_export(directory)
    rows,excluded,report=reconstruct_swaps(events,checkpoint)
    write_csv(directory/"real_swaps.csv",rows,FIELDS)
    write_csv(directory/"excluded_events.csv",excluded,["block_number","log_index","reason"])
    (directory/"reconciliation.json").write_text(json.dumps(report,indent=2))
    return meta,rows,report


## Matching reference candles and source hashes

Reference availability is candle close plus a declared delay, not candle open. Both millisecond and microsecond archive timestamps are handled. Require a matching ETH quote-currency symbol; do not substitute ETH/USDT for an ETH/USDC pool without accounting for the basis. The source archives and their official SHA256 checksums are retained. Historical pair coverage must exist for the selected window.


In [4]:
def download_reference(directory, rows, meta):
    if not rows: raise ValueError("No retained swaps")
    directory=Path(directory); archive_dir=directory/"reference_archives"; archive_dir.mkdir(exist_ok=True)
    first=min(r["origin_timestamp"] for r in rows)-300; last=max(r["timestamp"] for r in rows)
    month=datetime.fromtimestamp(first,timezone.utc).replace(day=1,hour=0,minute=0,second=0,microsecond=0)
    end=datetime.fromtimestamp(last,timezone.utc); reference={}; sources=[]
    while month<=end:
        symbol=meta["reference_symbol"]; name=f"{symbol}-1m-{month:%Y-%m}.zip"
        url=f"https://data.binance.vision/data/spot/monthly/klines/{symbol}/1m/{name}"
        path=archive_dir/name
        if not path.exists():
            with urllib.request.urlopen(url,timeout=60) as response: payload=response.read()
            with urllib.request.urlopen(url+".CHECKSUM",timeout=30) as response: checksum=response.read().decode().split()[0]
            if hashlib.sha256(payload).hexdigest()!=checksum: raise ValueError("Reference archive checksum mismatch")
            path.write_bytes(payload); (archive_dir/(name+".CHECKSUM")).write_text(checksum+'  '+name+'\n')
        checksum=(archive_dir/(name+".CHECKSUM")).read_text().split()[0]
        if hashlib.sha256(path.read_bytes()).hexdigest()!=checksum: raise ValueError("Cached reference checksum mismatch")
        with zipfile.ZipFile(path) as archive:
            for member in archive.namelist():
                for row in csv.reader(io.TextIOWrapper(archive.open(member))):
                    close=int(row[6]); available=close/(1e6 if close>=10**14 else 1e3)+meta["reference_delay_seconds"]
                    if first<=available<=last:
                        price=float(row[4])
                        if not math.isfinite(price) or price<=0: raise ValueError("Invalid reference price")
                        reference[available]=price
        sources.append(url)
        month=(month.replace(day=28)+timedelta(days=4)).replace(day=1)
    write_csv(directory/"reference_prices.csv",[dict(available_at=k,price=v) for k,v in sorted(reference.items())],["available_at","price"])
    meta["reference_sources"]=sources; meta["reference_quote_symbol"]=meta["quote_symbol"]
    (directory/"dataset_card.json").write_text(json.dumps(meta,indent=2))

def model_events(rows, reference, meta, max_age=300):
    if meta.get("reference_quote_symbol") != meta.get("quote_symbol"): raise ValueError("Reference quote currency mismatch")
    base=int(meta["base_token_index"]); quote=1-base
    decimals=[int(meta[f"token{j}_decimals"]) for j in (0,1)]
    if base not in (0,1) or any(not 0<=d<=36 for d in decimals): raise ValueError("Invalid token metadata")
    reference=sorted(reference,key=lambda r:float(r["available_at"]))
    times=[float(r["available_at"]) for r in reference]
    if len(set(times))!=len(times): raise ValueError("Duplicate reference availability times")
    if any(not math.isfinite(t) for t in times): raise ValueError("Invalid reference times")
    output=[]; counts=Counter(); segment=-1; previous=None; previous_ref=None; force_split=True
    for raw in rows:
        row=dict(raw); origin=float(row["origin_timestamp"]); time=float(row["timestamp"])
        if not math.isfinite(origin) or not math.isfinite(time) or origin>time: raise ValueError("Invalid origin time")
        index=bisect_left(times,origin)-1
        if index<0 or origin-times[index]>max_age:
            counts["missing_or_stale_reference"]+=1; force_split=True; continue
        price=float(reference[index]["price"])
        if not math.isfinite(price) or price<=0: raise ValueError("Invalid reference price")
        before=[int(row[f"reserve{j}_before"]) for j in (0,1)]
        after=[int(row[f"reserve{j}_after"]) for j in (0,1)]
        split=force_split or previous is None or str(raw["segment"])!=str(previous["segment"]) or before!=previous["after"]
        if not split and origin<previous["timestamp"]: raise ValueError("Origin precedes observed state")
        if split: segment+=1; previous_ref=None
        reference_coordinate=math.log(price)
        amount_base,amount_quote=int(row[f"amount{base}In"]),int(row[f"amount{quote}In"])
        u=amount_quote/before[quote] if amount_quote else -amount_base/before[base]
        if min(before+after)<=0 or not math.isfinite(u): raise ValueError("Invalid model state or flow")
        output.append(dict(timestamp=time,origin_timestamp=origin,block_number=int(row["block_number"]),
                           transaction_index=int(row["transaction_index"]),log_index=int(row["log_index"]),segment=segment,
                           state_a_before=before[base]/10**decimals[base],state_b_before=before[quote]/10**decimals[quote],
                           state_a_after=after[base]/10**decimals[base],state_b_after=after[quote]/10**decimals[quote],flow=u,
                           reference_coordinate=reference_coordinate,
                           reference_return=0. if previous_ref is None else reference_coordinate-previous_ref,
                           reference_available_at=times[index]))
        previous=dict(segment=raw["segment"],after=after,timestamp=time); previous_ref=reference_coordinate; force_split=False
    counts.update(retained=len(output),input_swaps=len(rows),segments=segment+1)
    return output,dict(counts)

def finalize_export(directory, rows, meta):
    directory=Path(directory)
    with (directory/"reference_prices.csv").open() as f: reference=list(csv.DictReader(f))
    normalized,report=model_events(rows,reference,meta)
    if not normalized: raise ValueError("No model events remain after reference alignment")
    write_csv(directory/"model_events.csv",normalized,list(normalized[0]))
    (directory/"model_quality.json").write_text(json.dumps(report,indent=2))
    hashes={str(p.relative_to(directory)):hashlib.sha256(p.read_bytes()).hexdigest()
            for p in directory.rglob("*") if p.is_file() and p.name!="checksums.json"}
    (directory/"checksums.json").write_text(json.dumps(hashes,indent=2))

if ACQUIRE:
    # Pilot covers the first three days only; no model performance is inspected.
    if not (DATA_DIR/"pilot_report.json").exists() and not (DATA_DIR/"checksums.json").exists():
        pilot_end=first_block_at("2024-01-04T00:00:00+00:00")-1
        acquire_export(DATA_DIR,stop_block=pilot_end)
        metadata,swaps,pilot=prepare_export(DATA_DIR)
        (DATA_DIR/"pilot_report.json").write_text(json.dumps(pilot,indent=2))
        print("Three-day reconciliation pilot:",pilot,flush=True)
        if not swaps: raise ValueError("Pilot retained no ordinary swaps")
    frozen=(DATA_DIR/"checksums.json").exists()
    acquire_export(DATA_DIR)
    if not frozen:
        metadata,swaps,report=prepare_export(DATA_DIR)
        download_reference(DATA_DIR,swaps,metadata)
        finalize_export(DATA_DIR,swaps,metadata)
        print("Prepared real export:",report,flush=True)
else:
    print("Live acquisition disabled. Set AMM_ACQUIRE=1 to acquire the frozen real window.")


Acquisition: 32 / 39914 ranges; last block 18909205


Acquisition: 320 / 39914 ranges; last block 18912085


Acquisition: 608 / 39914 ranges; last block 18914965


Acquisition: 896 / 39914 ranges; last block 18917845


Acquisition: 1184 / 39914 ranges; last block 18920725


Acquisition: 1472 / 39914 ranges; last block 18923605


... (133 outputs left out)

Acquisition: 39352 / 39914 ranges; last block 19302405


Acquisition: 39640 / 39914 ranges; last block 19305285


Acquisition: 39914 / 39914 ranges; last block 19308025


Prepared real export: {'swaps_seen': 129289, 'retained': 126934, 'nonstandard_output_quote': 2352, 'non_swap_sync': 113, 'nonordinary_swap': 3, 'retained_fraction': 0.9817849933095623}


## Offline fixture checks

These small records are generated solely to exercise reconstruction, rounding, event ordering, liquidity segmentation and rejected inputs. Their output directory and dataset card explicitly say `synthetic_fixture`. They are not real trades or empirical results. Run these assertions even when network acquisition is unavailable.


In [5]:
def fixture_events(n=400):
    rng=random.Random(19); reserves=(10**12,5*10**20); checkpoint=dict(reserves=reserves,timestamp=1700000000,block=0)
    events=[]; reference=[]
    for i in range(n):
        timestamp=checkpoint["timestamp"]+86400*(i//8)+12*(i%8)+12
        if i==n//2:
            reserves=tuple(v*2 for v in reserves)
            events.append(dict(kind="Sync",values=list(reserves),timestamp=timestamp,block_number=i+1,
                               transaction_index=0,log_index=0,transaction_hash=f"liquidity-{i}"))
        r0,r1=reserves; direction=rng.randrange(2)
        amount=max(1,int(reserves[direction]*(.001+.002*rng.random())))
        if direction==0:
            output=amount*997*r1//(1000*r0+997*amount); amounts=(amount,0,0,output)
        else:
            output=amount*997*r0//(1000*r1+997*amount); amounts=(0,amount,output,0)
        i0,i1,o0,o1=amounts; after=(r0+i0-o0,r1+i1-o1)
        base=dict(timestamp=timestamp,block_number=i+1,transaction_index=1,transaction_hash=f"swap-{i}")
        events += [dict(base,kind="Sync",values=list(after),log_index=1),dict(base,kind="Swap",values=list(amounts),log_index=2)]
        reference.append(dict(available_at=timestamp-1,price=(r0/10**6)/(r1/10**18)))
        reserves=after
    return checkpoint,events,reference

checkpoint,events,reference=fixture_events()
fixture_rows,excluded,report=reconstruct_swaps(events,checkpoint)
assert len(fixture_rows)==400 and not excluded and len({r["segment"] for r in fixture_rows})==2
assert fixture_rows[0]["reserve0_before"]==checkpoint["reserves"][0]
assert fixture_rows[0]["origin_timestamp"]==checkpoint["timestamp"]
try:
    reconstruct_swaps(events+[events[0]],checkpoint)
    raise AssertionError("Duplicate events were accepted")
except ValueError: pass
altered=[dict(e,values=list(e["values"])) for e in events]
altered[1]["values"][3]+=1
assert reconstruct_swaps(altered,checkpoint)[2]["reserve_accounting"]==1
fixture_dir=Path(tempfile.mkdtemp(prefix="amm_synthetic_fixture_"))
write_csv(fixture_dir/"real_swaps.csv",fixture_rows,FIELDS)
# A pre-window quote is needed for the first origin; prices are synthetic test values.
reference.insert(0,dict(available_at=checkpoint["timestamp"]-1,price=2000.))
write_csv(fixture_dir/"reference_prices.csv",reference,["available_at","price"])
fixture_meta=dict(data_kind="synthetic_fixture",pool_address="fixture_only",base_token_index=1,
                  token0_decimals=6,token1_decimals=18,fee=.003,quote_symbol="USDC",reference_quote_symbol="USDC",
                  reference_symbol="ETHUSDC",reference_delay_seconds=0,source="deterministic offline fixture")
(fixture_dir/"dataset_card.json").write_text(json.dumps(fixture_meta,indent=2))
normalized,quality=model_events(fixture_rows,reference,fixture_meta)
assert len(normalized)==400 and quality["segments"]==2
assert math.isclose(normalized[0]["state_b_before"]/normalized[0]["state_a_before"],2000.)
assert all(r["reference_available_at"]<r["origin_timestamp"] for r in normalized)
assert all(math.isfinite(r["flow"]) for r in normalized)
future_reference=[dict(available_at=checkpoint["timestamp"]+10**9,price=2000.)]
assert not model_events(fixture_rows,future_reference,fixture_meta)[0]
try:
    model_events(fixture_rows,reference,dict(fixture_meta,reference_quote_symbol="USDT"))
    raise AssertionError("Mismatched reference quote was accepted")
except ValueError: pass
finalize_export(fixture_dir,fixture_rows,fixture_meta)
print("Offline reconstruction checks passed; fixture only:",fixture_dir)


Offline reconstruction checks passed; fixture only: /tmp/amm_synthetic_fixture_tjy1fn3q
